In [ ]:

# Cell 1: Install required libraries

# !pip -q install transformers scikit-learn nltk pandas numpy
!pip -q install rouge-score bert-score


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
%cd "/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum"

/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum


In [ ]:
%ls

1-Extractive-Summarizer.ipynb  multiclinsum_large-scale_train_en/
multiclinsum_gs_train_en/      multiclinsum_test_en/


In [ ]:

# Cell 2: Import libraries and load BERT

import torch
import numpy as np
import pandas as pd

from transformers import AutoTokenizer, AutoModel
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_distances

MODEL_NAME = "bert-base-multilingual-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print("Device:", device)
print("Model loaded:", MODEL_NAME)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
Model loaded: bert-base-multilingual-cased


In [ ]:
# Reading the dataset

from pathlib import Path
import pandas as pd

BASE_DIR = Path(
    "multiclinsum_gs_train_en/"
)

FULLTEXT_DIR = BASE_DIR / "fulltext"
SUMMARY_DIR = BASE_DIR / "summaries"

# Load full-text reports
reports = {
    path.stem: path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()
    for path in FULLTEXT_DIR.glob("*.txt")
}

# Load summaries, removing the "_sum" suffix
summaries = {
    path.stem.removesuffix("_sum"): path.read_text(
        encoding="utf-8",
        errors="replace"
    ).strip()
    for path in SUMMARY_DIR.glob("*.txt")
}

# Match reports and summaries by their base filename
common_ids = sorted(
    set(reports.keys()) & set(summaries.keys())
)

df = pd.DataFrame([
    {
        "filename": file_id + ".txt",
        "report": reports[file_id],
        "summary": summaries[file_id]
    }
    for file_id in common_ids
])

print("Full-text files:", len(reports))
print("Summary files:", len(summaries))
print("Matched pairs:", len(df))

display(df.head(3))


Full-text files: 592
Summary files: 592
Matched pairs: 592


,filename,report,summary
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...
2,multiclinsum_gs_en_100.txt,A six-month-old male infant was observed in th...,A six-month old infant presented to the emerge...


In [ ]:
# Check for unmatched files


unmatched_reports = set(reports) - set(summaries)
unmatched_summaries = set(summaries) - set(reports)

print("Unmatched reports:", len(unmatched_reports))
print("Unmatched summaries:", len(unmatched_summaries))

print("Empty reports:", (df["report"].str.len() == 0).sum())
print("Empty summaries:", (df["summary"].str.len() == 0).sum())

assert len(df) > 0, "No matching report-summary pairs found!"


Unmatched reports: 0
Unmatched summaries: 0
Empty reports: 0
Empty summaries: 0


In [ ]:
# print("Total matched pairs:", len(df))
# print("Unique filenames:", df["filename"].nunique())

# assert len(df) == 592, "Expected 592 gold-standard pairs"
# assert df["filename"].nunique() == len(df)

# print("Dataset validation passed!")

Total matched pairs: 592
Unique filenames: 592
Dataset validation passed!


# Pre-processing

In [ ]:
# import re
# text = "5.6"
# print(text)
# text = re.sub(
#         r'(?<!\w)(\d+)\.(\d+)(?!\w)',
#         r'\1DOT\2',
#         text
#     )
# print(text)

5.6
5DOT6


In [ ]:
# # 3. Standardize phrases
# text = "and/or"
# print(text)
# text = text.replace(text, "and or")
# print(text)

and/or
and or


In [ ]:
# # 4. Separate hyphenated number-word patterns
#   # Examples: 52-year-old -> 52 year old
#   #           10-day history -> 10 day history
#   #           3-month follow-up -> 3 month follow up

# text = "5-year-old"
# text = "follow-up"
# print(text)
# text = re.sub(
#         r'(?<=\d)-(?=[a-z])|(?<=[a-z])-(?=[a-z])',
#         ' ',
#         text
#     )
# print(text)

follow-up
follow up


In [ ]:
# # 5. Remove unnecessary spaces and normalize whitespace
# text = " Hello   this is     just for   the           testing."
# print(text)
# text = re.sub(r'\s+', ' ', text).strip()
# print(text)


 Hello   this is     just for   the           testing.
Hello this is just for the testing.


In [ ]:
# define a function for preprocessing with all steps executed at once

import re
import pandas as pd

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Convert all characters to lowercase
    text = text.lower()

    # 2. Replace decimal points with a temporary placeholder
    #    Example: 5.3 -> 5DOT3
    text = re.sub(
        r'(?<!\w)(\d+)\.(\d+)(?!\w)',
        r'\1DOT\2',
        text
    )

    # 3. Standardize phrases
    text = text.replace("and/or", "and or")

    # 4. Separate hyphenated number-word patterns
    # Examples: 52-year-old -> 52 year old
    #           10-day history -> 10 day history
    #           3-month follow-up -> 3 month follow up
    text = re.sub(
        r'(?<=\d)-(?=[a-z])|(?<=[a-z])-(?=[a-z])',
        ' ',
        text
    )

    # 5. Remove unnecessary spaces and normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# save a new dataframe
cleaned_df = df.copy(deep=True)

# Apply preprocessing
cleaned_df["cleaned_report"] = df["report"].apply(preprocess_text)
cleaned_df["cleaned_summary"] = df["summary"].apply(preprocess_text)

print("Preprocessing completed!")


Preprocessing completed!


In [ ]:
df.head(2)

,filename,report,summary
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...


In [ ]:
cleaned_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...


In [ ]:
cleaned_df['cleaned_report'][0]

'a 52 year old female patient from wolkiet (northwest ethiopia) was apparently healthy two years ago but then started experiencing irritability, heat intolerance and diaphoresis. she had an anterior neck swelling of 30 years’ duration. for this, she was evaluated at haider referral hospital (north ethiopia), diagnosed with a toxic multinodular goiter and managed with propylthiouracil 100 mg orally (po) three times per day. six months following initiation of the medication, her symptoms improved, and she was offered surgical treatment, which she refused. four months prior to her current presentation, she developed generalized body fatigue which worsened during activities like climbing stairs, combing her hair and standing from a squatting position. concomitantly, she noticed violaceous rash over both hands and neck, which later involved the anterior part of her bilateral thighs and shoulders, hair loss and facial puffiness, but no morning stiffness. one week preceding her current visit,

# Sentence Segmentation

In [ ]:

!pip -q install nltk

import nltk
nltk.download("punkt_tab")

from nltk.tokenize import sent_tokenize


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
# # Testing on one sample

# # Select one example report
# processed_text = cleaned_df["cleaned_report"][0]

# sentences = sent_tokenize(processed_text, language="english")

# print("Number of sentences:", len(sentences))

# for i, sentence in enumerate(sentences, start=1):
#     print(f"{i}. {sentence}")


Number of sentences: 37
1. a 52 year old female patient from wolkiet (northwest ethiopia) was apparently healthy two years ago but then started experiencing irritability, heat intolerance and diaphoresis.
2. she had an anterior neck swelling of 30 years’ duration.
3. for this, she was evaluated at haider referral hospital (north ethiopia), diagnosed with a toxic multinodular goiter and managed with propylthiouracil 100 mg orally (po) three times per day.
4. six months following initiation of the medication, her symptoms improved, and she was offered surgical treatment, which she refused.
5. four months prior to her current presentation, she developed generalized body fatigue which worsened during activities like climbing stairs, combing her hair and standing from a squatting position.
6. concomitantly, she noticed violaceous rash over both hands and neck, which later involved the anterior part of her bilateral thighs and shoulders, hair loss and facial puffiness, but no morning stiffne

In [ ]:

sent_seg_df = cleaned_df.copy(deep=True)

# Apply sentence segmentation to all reports and sumamries
sent_seg_df["r_sentences"] = cleaned_df["cleaned_report"].apply(
    lambda text: sent_tokenize(text, language="english")
)

sent_seg_df["su_sentences"] = cleaned_df["cleaned_summary"].apply(
    lambda text: sent_tokenize(text, language="english")
)

sent_seg_df["r_sent_count"] = sent_seg_df["r_sentences"].apply(len)
sent_seg_df["su_sent_count"] = sent_seg_df["su_sentences"].apply(len)

print("Average sentences per report:", sent_seg_df["r_sent_count"].mean())
print("Maximum sentences:", sent_seg_df["r_sent_count"].max())
print("Minimum sentences:", sent_seg_df["r_sent_count"].min())

print("Average sentences per summary:", sent_seg_df["su_sent_count"].mean())
print("Maximum sentences:", sent_seg_df["su_sent_count"].max())
print("Minimum sentences:", sent_seg_df["su_sent_count"].min())


Average sentences per report: 27.052364864864863
Maximum sentences: 198
Minimum sentences: 4
Average sentences per summary: 5.484797297297297
Maximum sentences: 30
Minimum sentences: 1


In [ ]:
sent_seg_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,[a 52 year old female patient from wolkiet (no...,[a 52 year old known female patient with a tox...,37,5
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...,[a 61 year old patient with steroid therapy fo...,[this case report is about a 61 year old patie...,28,2


In [ ]:
sent_seg_df["su_sentences"][0]

["a 52 year old known female patient with a toxic multi nodular goiter presented with insidious onset of itchy erythematous rash on her neck and bilateral hands and progressive muscle weakness of 4 months' duration.",
 "associated with this, she had photosensitivity, and periorbital edema of 2 months' duration.",
 'on physical examination, she had an anterior neck mass, proximal muscle weakness, periorbital edema, and violaceous skin rash on her bilateral arms, shoulders and neck.',
 'thyroid function tests were normal, creatinine kinase was elevated, and muscle biopsy revealed inflammatory myositis.',
 'ultrasound of the anterior neck mass and analysis of fine needle aspiration suggested thyroid cancer.']

In [ ]:
# Saving the latest df: sent_seg_df

from pathlib import Path

# Create an output folder in Google Drive
OUTPUT_DIR = Path.cwd()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save the dataframe
csv_path = OUTPUT_DIR / "sentence_segmented_gs_train_en.csv"

sent_seg_df.to_csv(
    csv_path,
    index=False,
    encoding="utf-8"
)

print("Saved successfully!")
print("File path:", csv_path)
print("Number of reports:", len(sent_seg_df))

Saved successfully!
File path: /content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/sentence_segmented_gs_train_en.csv
Number of reports: 592


In [ ]:
# Anytime we can directly import the latest dataframe (sent_seg_df) which is already saved for further steps

import pandas as pd
sent_seg_df = pd.read_csv("/content/drive/MyDrive/PhD_Documents/5-PhD-July-2026-IMP-Info/2-IITG-Classes/2-NLP-with-LLM/MultiClinSum/sentence_segmented_gs_train_en.csv")


In [ ]:
sent_seg_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,['a 52 year old female patient from wolkiet (n...,"[""a 52 year old known female patient with a to...",37,5
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...,['a 61 year old patient with steroid therapy f...,['this case report is about a 61 year old pati...,28,2


In [ ]:
# Load pretrained BERT

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "bert-base-multilingual-cased"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Device:", device)
print("BERT model loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda
BERT model loaded: bert-base-multilingual-cased


# BERT Sentence Embeddings Generation

In [ ]:

def encode_sentences(sentences, batch_size=16, max_length=256):
    """
    Convert sentences into BERT embeddings.

    Returns:
        NumPy array of shape (number_of_sentences, 768)
    """
    if not sentences:
        return np.empty((0, model.config.hidden_size), dtype=np.float32)

    all_embeddings = []

    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():
            outputs = model(**inputs)
            token_embeddings = outputs.last_hidden_state

            # Expand attention mask to match embedding dimensions
            mask = inputs["attention_mask"].unsqueeze(-1)
            mask = mask.to(token_embeddings.dtype)

            # Mean pooling, excluding padding tokens
            summed = (token_embeddings * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1)

            sentence_embeddings = summed / counts

        all_embeddings.append(
            sentence_embeddings.cpu().numpy()
        )

    return np.concatenate(all_embeddings, axis=0)


In [ ]:
import ast

sent_seg_df["r_sentences"] = sent_seg_df["r_sentences"].apply(ast.literal_eval)
sent_seg_df["su_sentences"] = sent_seg_df["su_sentences"].apply(ast.literal_eval)


In [ ]:
print(type(sent_seg_df.loc[0, "r_sentences"]))
print(len(sent_seg_df.loc[0, "r_sentences"]))
print(sent_seg_df.loc[0, "r_sentences"][0])

<class 'list'>
37
a 52 year old female patient from wolkiet (northwest ethiopia) was apparently healthy two years ago but then started experiencing irritability, heat intolerance and diaphoresis.


In [ ]:
# Test on one report first

sample_sentences = sent_seg_df["r_sentences"][0]

sample_embeddings = encode_sentences(sample_sentences)

print("Number of sentences:", len(sample_sentences))
print("Embedding shape:", sample_embeddings.shape)
print("Embedding dtype:", sample_embeddings.dtype)
print("First sentence:", sample_sentences[0])
print("First embedding (first 10 values):", sample_embeddings[0][:10])


Number of sentences: 37
Embedding shape: (37, 768)
Embedding dtype: float32
First sentence: a 52 year old female patient from wolkiet (northwest ethiopia) was apparently healthy two years ago but then started experiencing irritability, heat intolerance and diaphoresis.
First embedding (first 10 values): [-0.36722445 -0.59499186  0.3624427   0.19595759  0.27594534 -0.17281778
 -0.4328117   0.23055896 -0.17297551 -0.15494815]


In [ ]:
# Generate embeddings for all the reports

# separate dataframe
bert_df = sent_seg_df.copy(deep=True)

all_report_embeddings = []

for i, sentences in enumerate(bert_df["r_sentences"]):
    embeddings = encode_sentences(
        sentences,
        batch_size=16,
        max_length=256
    )

    all_report_embeddings.append(embeddings)

    if (i + 1) % 25 == 0 or i == len(bert_df) - 1:
        print(f"Processed {i + 1}/{len(bert_df)} reports")

bert_df["sentence_embeddings"] = all_report_embeddings

print("\nEmbedding generation completed!")
print("Total reports:", len(bert_df))
print("First report embedding shape:", bert_df["sentence_embeddings"][0].shape)


Processed 25/592 reports
Processed 50/592 reports
Processed 75/592 reports
Processed 100/592 reports
Processed 125/592 reports
Processed 150/592 reports
Processed 175/592 reports
Processed 200/592 reports
Processed 225/592 reports
Processed 250/592 reports
Processed 275/592 reports
Processed 300/592 reports
Processed 325/592 reports
Processed 350/592 reports
Processed 375/592 reports
Processed 400/592 reports
Processed 425/592 reports
Processed 450/592 reports
Processed 475/592 reports
Processed 500/592 reports
Processed 525/592 reports
Processed 550/592 reports
Processed 575/592 reports
Processed 592/592 reports

Embedding generation completed!
Total reports: 592
First report embedding shape: (37, 768)


In [ ]:
bert_df["sentence_embeddings"][0]

array([[-0.36722445, -0.59499186,  0.3624427 , ...,  0.5791843 ,
         0.17465575,  0.03734263],
       [-0.16591644, -0.07551321,  0.4433679 , ...,  0.9446057 ,
         0.0504608 , -0.293466  ],
       [-0.28572524, -0.26366812, -0.05760938, ...,  0.5565871 ,
         0.16619843, -0.20028575],
       ...,
       [-0.28054172, -0.13071688,  0.40461725, ...,  0.4995133 ,
         0.13761136, -0.19160874],
       [-0.42464632, -0.25933117, -0.04301205, ...,  0.6306895 ,
         0.05227759,  0.08276617],
       [-0.4763728 , -0.1837043 ,  0.08127747, ...,  0.34722942,
         0.13173473, -0.08392723]], dtype=float32)

# K-means clustering

In [ ]:

from sklearn.cluster import KMeans
import numpy as np

def cluster_and_select_sentences(sentences, embeddings, n_clusters=10):
    """
    Select representative sentences using K-means centroids.

    Returns selected sentences in original document order.
    """
    n_sentences = len(sentences)

    if n_sentences == 0:
        return []

    # Handle reports containing fewer than 10 sentences
    k = min(n_clusters, n_sentences)

    # If K equals the number of sentences, retain all sentences
    if k == n_sentences:
        return list(sentences)

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(embeddings)
    centroids = kmeans.cluster_centers_

    selected_indices = []

    for cluster_id in range(k):
        # Find sentence indices belonging to this cluster
        cluster_indices = np.where(labels == cluster_id)[0]

        if len(cluster_indices) == 0:
            continue

        # Select the sentence nearest to the cluster centroid
        cluster_embeddings = embeddings[cluster_indices]
        distances = np.linalg.norm(
            cluster_embeddings - centroids[cluster_id],
            axis=1
        )

        closest_index = cluster_indices[np.argmin(distances)]
        selected_indices.append(closest_index)

    # Restore original document order
    selected_indices = sorted(set(selected_indices))

    return [sentences[i] for i in selected_indices]


In [ ]:
bert_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count,sentence_embeddings
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,[a 52 year old female patient from wolkiet (no...,[a 52 year old known female patient with a tox...,37,5,"[[-0.36722445, -0.59499186, 0.3624427, 0.19595..."
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...,[a 61 year old patient with steroid therapy fo...,[this case report is about a 61 year old patie...,28,2,"[[-0.49747035, -0.50360686, 0.390913, 0.251491..."


In [ ]:

# Test on the first report
sample_sentences = bert_df["r_sentences"][0]
sample_embeddings = bert_df["sentence_embeddings"][0]

sample_selected = cluster_and_select_sentences(
    sample_sentences,
    sample_embeddings,
    n_clusters=10
)

print("Original sentence count:", len(sample_sentences))
print("Selected sentence count:", len(sample_selected))

print("\nExtractive summary:")
print(sample_selected)

Original sentence count: 37
Selected sentence count: 10

Extractive summary:
['she had an anterior neck swelling of 30 years’ duration.', 'six months following initiation of the medication, her symptoms improved, and she was offered surgical treatment, which she refused.', 'on physical examination, she was acutely sick looking, conscious and oriented.', 'the chest was resonant on percussion note and clear on auscultation.', 'abdominal examination revealed no enlarged organ or any sign of fluid collection and she had no visible deformity or limitation of movement on locomotor examination.', 'additionally, she had a hypopigmented area on her nails, hands, anterior chest and bilateral thigh.', 'her laboratory test results showed that she had moderate anemia (hemoglobin = 10.9g/dl, mcv = 78DOT0 fl, and rdw: 13DOT5%), with normal other blood cell lines and erythrocyte sedimentation rate.', 'renal function tests and serum electrolytes were normal.', 'another immunological panel was not done 

In [ ]:
# Generate summaries for all 592 reports

# new dataframe
cluster_df = bert_df.copy(deep=True)

cluster_df["selected_sentences"] = [
    cluster_and_select_sentences(
        sentences,
        embeddings,
        n_clusters=10
    )
    for sentences, embeddings in zip(
        cluster_df["r_sentences"],
        cluster_df["sentence_embeddings"]
    )
]

cluster_df["generated_summary"] = (
    cluster_df["selected_sentences"].apply(" ".join)
)

print("Summaries generated:", len(cluster_df))
print("\nFirst generated summary:\n")
print(cluster_df["generated_summary"][0])


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (6) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


Summaries generated: 592

First generated summary:

she had an anterior neck swelling of 30 years’ duration. six months following initiation of the medication, her symptoms improved, and she was offered surgical treatment, which she refused. on physical examination, she was acutely sick looking, conscious and oriented. the chest was resonant on percussion note and clear on auscultation. abdominal examination revealed no enlarged organ or any sign of fluid collection and she had no visible deformity or limitation of movement on locomotor examination. additionally, she had a hypopigmented area on her nails, hands, anterior chest and bilateral thigh. her laboratory test results showed that she had moderate anemia (hemoglobin = 10.9g/dl, mcv = 78DOT0 fl, and rdw: 13DOT5%), with normal other blood cell lines and erythrocyte sedimentation rate. renal function tests and serum electrolytes were normal. another immunological panel was not done due to limited clinical set up. chest x ray was nor

In [ ]:
cluster_df.head(2)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count,sentence_embeddings,selected_sentences,generated_summary
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,[a 52 year old female patient from wolkiet (no...,[a 52 year old known female patient with a tox...,37,5,"[[-0.36722445, -0.59499186, 0.3624427, 0.19595...",[she had an anterior neck swelling of 30 years...,she had an anterior neck swelling of 30 years’...
1,multiclinsum_gs_en_10.txt,A 61-year-old patient with steroid therapy for...,This case report is about a 61-year-old patien...,a 61 year old patient with steroid therapy for...,this case report is about a 61 year old patien...,[a 61 year old patient with steroid therapy fo...,[this case report is about a 61 year old patie...,28,2,"[[-0.49747035, -0.50360686, 0.390913, 0.251491...",[joint aspiration showed pus in both hip joint...,joint aspiration showed pus in both hip joints...


In [ ]:
type(cluster_df["cleaned_summary"][0])

str

In [ ]:
type(cluster_df["generated_summary"][0])

str

In [ ]:
type(cluster_df["su_sentences"][0])

list

In [ ]:
print("Generated summary:")
print(cluster_df["generated_summary"][0])

print("\nHuman-written reference summary:")
print(cluster_df["cleaned_summary"][0])

Generated summary:
she had an anterior neck swelling of 30 years’ duration. six months following initiation of the medication, her symptoms improved, and she was offered surgical treatment, which she refused. on physical examination, she was acutely sick looking, conscious and oriented. the chest was resonant on percussion note and clear on auscultation. abdominal examination revealed no enlarged organ or any sign of fluid collection and she had no visible deformity or limitation of movement on locomotor examination. additionally, she had a hypopigmented area on her nails, hands, anterior chest and bilateral thigh. her laboratory test results showed that she had moderate anemia (hemoglobin = 10.9g/dl, mcv = 78DOT0 fl, and rdw: 13DOT5%), with normal other blood cell lines and erythrocyte sedimentation rate. renal function tests and serum electrolytes were normal. another immunological panel was not done due to limited clinical set up. chest x ray was normal.

Human-written reference sum

In [ ]:
# Install evaluation libraries

!pip -q install rouge-score bert-score

In [ ]:
cluster_df.head(1)

,filename,report,summary,cleaned_report,cleaned_summary,r_sentences,su_sentences,r_sent_count,su_sent_count,sentence_embeddings,selected_sentences,generated_summary
0,multiclinsum_gs_en_1.txt,A 52-year-old female patient from Wolkiet (Nor...,A 52-year-old known female patient with a toxi...,a 52 year old female patient from wolkiet (nor...,a 52 year old known female patient with a toxi...,[a 52 year old female patient from wolkiet (no...,[a 52 year old known female patient with a tox...,37,5,"[[-0.36722445, -0.59499186, 0.3624427, 0.19595...",[she had an anterior neck swelling of 30 years...,she had an anterior neck swelling of 30 years’...


In [ ]:
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer

# Use the existing test predictions.
predictions = cluster_df["generated_summary"].fillna("").astype(str).tolist()

# IMPORTANT:
# Use the reference summaries in the SAME text representation
# as the summaries used for model training/evaluation.
references = cluster_df["cleaned_summary"].fillna("").astype(str).tolist()

assert len(predictions) == len(references)
assert len(predictions) > 0

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

all_scores = []

for prediction, reference in zip(predictions, references):
    # rouge_score expects reference first, prediction second.
    scores = scorer.score(reference, prediction)

    row = {}

    for metric in ["rouge1", "rouge2", "rougeL"]:
        row[f"{metric}_precision"] = scores[metric].precision
        row[f"{metric}_recall"] = scores[metric].recall
        row[f"{metric}_f1"] = scores[metric].fmeasure

    all_scores.append(row)

rouge_per_example = pd.DataFrame(all_scores)

# Mean of the individual example scores, on a 0–100 scale.
rouge_summary = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
    ],
    "Precision": [
        rouge_per_example["rouge1_precision"].mean() * 100,
        rouge_per_example["rouge2_precision"].mean() * 100,
        rouge_per_example["rougeL_precision"].mean() * 100
    ],
    "Recall": [
        rouge_per_example["rouge1_recall"].mean() * 100,
        rouge_per_example["rouge2_recall"].mean() * 100,
        rouge_per_example["rougeL_recall"].mean() * 100
    ],
    "F1": [
        rouge_per_example["rouge1_f1"].mean() * 100,
        rouge_per_example["rouge2_f1"].mean() * 100,
        rouge_per_example["rougeL_f1"].mean() * 100
    ]
})

print("ROUGE RESULTS (mean per-example scores, %)")
display(rouge_summary.round(4))

ROUGE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,ROUGE-1,28.7717,55.1647,36.0883
1,ROUGE-2,11.3899,21.3543,14.1489
2,ROUGE-L,17.8836,34.8950,22.5397


In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate

bertscore = evaluate.load("bertscore")

bert_results = bertscore.compute(
    predictions=predictions,
    references=references,
    model_type="roberta-large",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=4,
    rescale_with_baseline=False
)

bert_summary = pd.DataFrame({
    "Metric": ["BERTScore"],
    "Precision": [np.mean(bert_results["precision"]) * 100],
    "Recall": [np.mean(bert_results["recall"]) * 100],
    "F1": [np.mean(bert_results["f1"]) * 100]
})

print("BERTSCORE RESULTS (mean per-example scores, %)")
display(bert_summary.round(4))

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTSCORE RESULTS (mean per-example scores, %)


,Metric,Precision,Recall,F1
0,BERTScore,82.8906,86.8404,84.7998
